<a href="https://colab.research.google.com/github/cckk-2021/python-basic-kadai/blob/fix2/%E3%82%B1%E3%83%BC%E3%82%B9%E3%82%B9%E3%82%BF%E3%83%87%E3%82%A33%E6%9C%80%E7%B5%82%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import glob
import os
from datetime import datetime

# ========== パス設定 ==========
BASE_DIR = r"/content/drive/MyDrive/samples"
INV_PATH = os.path.join(BASE_DIR, "inventory.xlsx")
ORDER_DIR = os.path.join(BASE_DIR, "order_new")
PICKUP_PATH = os.path.join(BASE_DIR, "pickup.xlsx")

# ========== inventory.xlsx 読み込み ==========
df_inv = pd.read_excel(INV_PATH)

# C列以降が野菜の列（A:日付, B:曜日）
items = df_inv.columns[2:]   # ['トマト','キャベツ','レタス',...]

# 最新在庫（最終行）を取得
if df_inv.shape[0] == 0:
    latest_stock = pd.Series(0, index=items)
else:
    # NaNを0に置き換えてからint型に変換
    latest_stock = df_inv.iloc[-1][items].fillna(0).astype(int).copy()

# ========== 各店舗の注文を集計（在庫減算用） ==========
order_files = glob.glob(os.path.join(ORDER_DIR, "order_*_20230524.xlsx"))

total_order = {item: 0 for item in items}

for file in order_files:
    df_order = pd.read_excel(file, header=0)  # 1行目: 列名, 2行目: データ
    order_row = df_order.iloc[0]
    for item in items:
        if item in order_row.index and not pd.isna(order_row[item]):
            total_order[item] += int(order_row[item])

# 注文反映後の在庫（＝最新在庫 － 注文数）
stock_after_orders = latest_stock.copy()
for item in items:
    stock_after_orders[item] = int(stock_after_orders[item]) - total_order[item]

# ========== pickup.xlsx 読み込み（しきい値＆発注量） ==========
df_pick = pd.read_excel(PICKUP_PATH, header=0)

# 1行目：しきい値, 2行目：追加量（B列以降が野菜）
threshold = df_pick.iloc[0][1:]  # index: トマト, キャベツ, ...
add_amount = df_pick.iloc[1][1:]

# ========== しきい値判定 → 発注 → 納品後在庫 ==========
final_stock = stock_after_orders.copy()
order_needed = {}

for item in items:
    # 注文反映後の在庫がしきい値未満なら発注
    if stock_after_orders[item] < threshold[item]:
        qty = int(add_amount[item])
        order_needed[item] = qty
        final_stock[item] = int(final_stock[item]) + qty
    else:
        final_stock[item] = int(final_stock[item])
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# ==========================
# Mailtrap SMTP 設定
# ==========================
SMTP_SERVER = "sandbox.smtp.mailtrap.io"
SMTP_PORT = 587
SMTP_USER = "c64fc9530c69d2"  # Mailtrapの値に置き換え
SMTP_PASS = "6bd70dffdb45ef"  # Mailtrapの値に置き換え

# ==========================
# メール送信先設定
# ==========================
sender_email = "noreply@example.com"           # 任意
receiver_email = "farmer@example.com"          # 取引先農家のメールアドレス（Mailtrap上ではダミー可）
subject = "【自動発注】野菜の追加注文について"

# ==========================
# メール本文生成
# ==========================

if len(order_needed) == 0:
    print("発注なしのため、メール送信は行いません。")
else:
    body_lines = []
    body_lines.append("以下の野菜について在庫がしきい値を下回ったため、自動発注いたします。\n")
    body_lines.append("【発注内容】\n")

    for item, qty in order_needed.items():
        body_lines.append(f"・{item}: {qty} 個")

    body_lines.append("\n本メールはシステムにより自動送信されています。")

    body = "\n".join(body_lines)

    # ==========================
    # MIME メール作成
    # ==========================
    msg = MIMEMultipart()
    msg["From"] = sender_email
    msg["To"] = receiver_email
    msg["Subject"] = subject

    msg.attach(MIMEText(body, "plain"))

    # ==========================
    # メール送信処理
    # ==========================
    try:
        with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
            server.starttls()
            server.login(SMTP_USER, SMTP_PASS)
            server.send_message(msg)

        print("発注メールを Mailtrap に送信しました。")

    except Exception as e:
        print("メール送信中にエラーが発生しました:")
        print(e)

# ========== inventory.xlsx に新しい行（日付・曜日＋最終在庫）を追加 ==========
from zoneinfo import ZoneInfo

# 時刻情報からタイムゾーン情報を削除
now = datetime.now(ZoneInfo("Asia/Tokyo")).replace(tzinfo=None)   # JSTで日時取得し、タイムゾーンを削除
weekday = now.strftime("%a")                 # 曜日（英語3文字）

new_row = {
    "日付": now,
    "曜日": weekday
}
for item in items:
    new_row[item] = int(final_stock[item])

df_inv = pd.concat([df_inv, pd.DataFrame([new_row])], ignore_index=True)
df_inv.to_excel(INV_PATH, index=False)


print("注文減算 → しきい値判定 → 発注 → 納品後在庫の更新まで完了しました。")

print("\n【各店舗合計の注文数量】")
for item in items:
    print(f"{item}: {total_order[item]}")

print("\n【発注された商品一覧】")
if not order_needed:
    print("今回は発注なし")
else:
    for item, qty in order_needed.items():
        print(f"{item}: {qty} 発注（在庫 {stock_after_orders[item]} → {final_stock[item]}）")

/tmp/ipython-input-1577527011.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  latest_stock = df_inv.iloc[-1][items].fillna(0).astype(int).copy()


発注メールを Mailtrap に送信しました。
注文減算 → しきい値判定 → 発注 → 納品後在庫の更新まで完了しました。

【各店舗合計の注文数量】
トマト: 31
キャベツ: 21
レタス: 42
白菜: 25
ほうれん草: 23
大根: 15
ニンジン: 32

【発注された商品一覧】
トマト: 100 発注（在庫 38 → 138）
キャベツ: 80 発注（在庫 38 → 118）
レタス: 100 発注（在庫 16 → 116）
白菜: 60 発注（在庫 10 → 70）
ほうれん草: 80 発注（在庫 34 → 114）
ニンジン: 80 発注（在庫 16 → 96）
